In [1]:
import json
import random
from openai import OpenAI
import anthropic
from dotenv import load_dotenv
import os
import re
import numpy as np
import base64
import time
import pandas as pd 
from tqdm import tqdm
from word2number import w2n

# Load dataset

In [2]:
# Set base directory using relative path
base_dir = os.path.join(os.getcwd(), "dataset", "simpsons")

# Set paths relative to base_dir
annotation_path = os.path.join(base_dir, "v1_Annotation_Val_simpsons_vqa.json")
question_path = os.path.join(base_dir, "v1_Question_Val_simpsons_vqa.json")
images_dir = os.path.join(base_dir, "val_images")

def load_dataset(annotation_path, question_path):
    try:
        with open(annotation_path, 'r') as f:
            annotations = json.load(f)['annotations']

        with open(question_path, 'r') as f:
            questions = json.load(f)['questions']

        # Select the dataset where ‘overall_scores’ == 1.0
        filtered_annotations = [
            annotation for annotation in annotations
            if annotation.get('overall_scores', {}).get('question') == 1.0 and
               annotation.get('overall_scores', {}).get('answer') == 1.0
        ]

        # Create a mapping from question ID to answer
        question_id_to_answer = {
            annotation['id']: {
                'answer': annotation['answer'],
                'answer_type': annotation['answer_type']  
            }
            for annotation in filtered_annotations
        }

        # Create a mapping from question ID to answer type
        question_id_to_answer_type = {
            annotation['id']: {
                'answer': annotation['answer'],
                'answer_type': annotation.get('answer_type', 'other')
            }
            for annotation in filtered_annotations
        }

        filtered_questions = [
            question for question in questions 
            if question['id'] in question_id_to_answer
        ]

        return filtered_questions, filtered_annotations, question_id_to_answer, question_id_to_answer_type

    except Exception as e:
        print(f"Error loading dataset: {e}")
        return [], [], {}, {}

def get_dataset(questions, question_id_to_answer, fraction=0.005, seed=42):
    # TODO：Increase quantity
# def get_dataset(questions, question_id_to_answer, fraction=0.05, seed=42):
    try:
        random.seed(seed)
        sample_size = max(1, int(len(questions) * fraction))
        sampled_questions = random.sample(questions, sample_size)
        sampled_truth_answers = [
            question_id_to_answer[q['id']]['answer'] 
            for q in sampled_questions
        ]
        
        for q in sampled_questions:
            q['answer_type'] = question_id_to_answer[q['id']]['answer_type']

        return sampled_questions, sampled_truth_answers

    except Exception as e:
        print(f"Error sampling dataset: {e}")
        return [], []

def encode_image(image_path):
    try:
        if not os.path.exists(image_path):
            print(f"Error: The image file at {image_path} was not found.")
            return None

        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')

    except Exception as e:
        print(f"An error occurred while encoding the image: {e}")
        return None

def parse_answer(input_str):
    if input_str is None:
        return None

    try:
        input_str = str(input_str).lower().strip()
        words = input_str.split()
        for i in range(len(words)):
            for j in range(i + 1, len(words) + 1):
                substring = ' '.join(words[i:j])
                try:
                    return str(w2n.word_to_num(substring))
                except:
                    continue

        matches = re.findall(r'\d+', input_str)
        if matches:
            return matches[-1]

        if "yes" in input_str:
            return "yes"
        elif "no" in input_str:
            return "no"

        return input_str

    except Exception as e:
        print(f"Error parsing answer '{input_str}': {e}")
        return input_str

evaluation_results = []


# Single agent prediction

In [3]:
load_dotenv()

# Configuration
# MODEL_NAME = "gpt-4o-mini"
MODEL_NAME = "claude-3-5-haiku-20241022"

is_openai_model = not MODEL_NAME.startswith("claude-")  

if is_openai_model:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    print(f"Using OpenAI model: {MODEL_NAME}")
else:
    client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
    print(f"Using Anthropic model: {MODEL_NAME}")

def get_predict(question, image_base64, max_retries=3, retry_delay=2):
    if image_base64 is None:
        return None

    prompt = f"""As a cartoon analysis expert, answer the question strictly based on the visual content and available context using EXACTLY ONE WORD:

    Input Question: {question}

    Guidelines:
    1. Consider cartoon-specific elements like character expressions, visual style, and narrative context.
    2. No explanations or punctuation allowed.
    """

    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME, 
                    messages=[
                        {
                            "role": "user",
                            "content": [
                                {"type": "text", "text": prompt},
                                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}},
                            ],
                        }
                    ],
                    max_tokens=150,
                    temperature=0.3,
                )
                response = completion.choices[0].message.content.strip().lower()
            else:
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user", 
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": image_base64}},
                        ]
                    }],
                    max_tokens=150,
                    temperature=0.3,
                )
                response = completion.content[0].text.strip().lower()

            # Process the response
            # 1. Remove punctuation marks
            response = response.rstrip('.!?')
            
            # 2. Split into words and get the first word
            words = response.split()
            if not words:
                continue
                
            first_word = words[0]
            
            # 3. Convert numbers
            try:
                # Check if word represents a number 
                number = w2n.word_to_num(first_word)
                return str(number)
            except ValueError:
                # Check if contains numeric digits
                matches = re.findall(r'\d+', first_word)
                if matches:
                    return matches[0]
                
                # If not number-related, return first word as is
                return first_word

        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            continue

    print(f"All {max_retries} attempts failed for question: {question}")
    return None


Using Anthropic model: claude-3-5-haiku-20241022


# Calculate accuracy

In [ ]:
def compute_accuracy(question, truth_answer, predicted_answer, answer_type, max_retries=2, retry_delay=2):
    if predicted_answer is None:
        return 0
    
    prompt = f"""
    Evaluate the accuracy of the predicted answer according to criteria below:

    Input:
    Question: {question}
    True answer: {truth_answer}
    Predicted answer: {predicted_answer}
    Answer type: {answer_type}

    Evaluation Rules:
    1. Be strict in your evaluation. The predicted answer must correctly address the question.
    2. Answers that claim "there is no information" or "there is no evidence" should be scored 0.0 when a definitive correct answer exists.
    3. Answers that contradict the correct answer should be scored 0.0.
    4. Answer Type Considerations:
    - Yes/No questions: Check if the meaning is equivalent
    - Number questions: Verify numerical accuracy
    - Other questions: Check for key information match

    2. Scoring Criteria:
    - 1.0: Contains the correct answer with the same core meaning as the reference
    - 0.75: Mostly correct with only minor differences that don't change the meaning
    - 0.5: Partially correct - contains some correct elements but misses important aspects
    - 0.25: Slightly correct - has a small element of the correct answer but is mostly wrong
    - 0.0: Completely incorrect, contradicts the correct answer, or avoids answering by claiming insufficient information

    Return only the numeric score (e.g. 0.75) with no explanation.
    """
    
    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=150,
                    temperature=0.3
                )
                response = completion.choices[0].message.content.strip()
            else:
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": prompt
                    }],
                    max_tokens=150,
                    temperature=0.3
                )
                response = completion.content[0].text.strip()

            numeric_match = re.search(r'(1\.0|0\.75|0\.5|0\.25|0\.0)', response)
            if numeric_match:
                score = float(numeric_match.group(1))
            else:
                score = 0.0

            return score

        except Exception as e:
            print(f"Accuracy calculation attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
    # Return 0 if it can't parse the score
    return 0.0


# Evaluate model performance

In [5]:
try:
    # Initialize counters
    correct_count = 0
    total_count = 0
    # Load dataset
    questions, annotations, question_id_to_answer, question_id_to_answer_type = load_dataset(annotation_path, question_path)

    if not questions:
        print("The 'questions' list is empty or not a list.")
        raise ValueError("Questions list is empty")

    # Get sample data TODO
    sampled_questions, sampled_truth_answers = get_dataset(questions, question_id_to_answer)

    if not sampled_questions:
        print("Failed to sample questions or empty sample")
        raise ValueError("No sampled questions")

    # Initialize results storage
    accuracies = []
    # Create an empty collection to store the processed problem IDs
    processed_question_ids = set() 

    # Process each question
    for question, truth_answer in tqdm(zip(sampled_questions, sampled_truth_answers),
                                     total=len(sampled_questions)):
        try:
            question_id = question['id']
            # Skip if already processed this question
            if question_id in processed_question_ids:
                continue
                
            processed_question_ids.add(question_id)
            
            question_text = question['question']
            image_relative_path = question['img_path']
            answer_type = question_id_to_answer_type[question_id]['answer_type']

            image_path = os.path.join(images_dir, image_relative_path)
            image_base64 = encode_image(image_path)

            if image_base64 is None:
                print(f"Skipping question ID {question_id} due to image encoding failure")
                continue

            model_answer = get_predict(question_text, image_base64)
            pred_solutions = [model_answer] if model_answer is not None else []

            if not pred_solutions:
                print(f"No prediction obtained for question ID {question_id}")
                continue

            accuracy = compute_accuracy(
                question=question_text,
                truth_answer=truth_answer,
                predicted_answer=model_answer,
                answer_type=answer_type
            )

            # Print results
            print(f"Question ID: {question_id}")
            print(f"Question: {question_text}")
            print(f"Answer Type: {answer_type}") 
            print(f"Truth Answer: {truth_answer}")
            if model_answer is not None:
                print(f"Predicted Answer: {model_answer}")
            if accuracy is not None:
                accuracies.append(accuracy)
                print(f"Accuracy: {accuracy:.4f}")
            else:
                print(f"Warning: No accuracy for question: {question_text}")

            # Store result
            result = {
                'question_id': question_id,
                'question': question_text,
                'answer_type': answer_type,
                'truth_answer': truth_answer,
                'predicted_answer': model_answer,
                'accuracy': accuracy
            }
            evaluation_results.append(result)
            accuracies.append(accuracy)

        except Exception as e:
            print(f"Error processing question {question.get('id', 'unknown')}: {e}")
            continue

    # Calculate average accuracy
    average_accuracy = np.mean(accuracies) if accuracies else 0
    print(f"Average Accuracy: {average_accuracy:.4f}")

except Exception as e:
    print(f"Unexpected error: {e}")
    average_accuracy = 0

  3%|▎         | 1/36 [00:03<02:00,  3.44s/it]

Question ID: 77311
Question: what is on the shelf?
Answer Type: other
Truth Answer: book
Predicted Answer: books
Accuracy: 0.7500


  6%|▌         | 2/36 [00:05<01:23,  2.45s/it]

Question ID: 12809
Question: how many people are in the picture?
Answer Type: number
Truth Answer: 1
Predicted Answer: 1
Accuracy: 1.0000


  8%|▊         | 3/36 [00:08<01:27,  2.65s/it]

...

Question ID: 71583
Question: what is in the background?
Answer Type: other
Truth Answer: table
Predicted Answer: boardgame
Accuracy: 0.2500


 97%|█████████▋| 35/36 [01:30<00:02,  2.40s/it]

Question ID: 37083
Question: what are the men doing?
Answer Type: other
Truth Answer: standing
Predicted Answer: waiting
Accuracy: 0.5000


100%|██████████| 36/36 [01:33<00:00,  2.59s/it]

Question ID: 96818
Question: what is the person sitting on?
Answer Type: other
Truth Answer: chair
Predicted Answer: chair
Accuracy: 1.0000
Average Accuracy: 0.8403


# Save results

In [6]:
# Clean up evaluation results to remove any existing average rows
evaluation_results = [r for r in evaluation_results if r['question_id'] != 'Average']

# Ensures no duplicate summary rows when saving results
unique_questions = len(set(r['question_id'] for r in evaluation_results))

# Add row numbers to each result
for i, result in enumerate(evaluation_results, 1):
    result['row_num'] = i

# Add average accuracy as the last row
average_result = {
    'row_num': len(evaluation_results) + 1,
    'question_id': 'Average',
    'question': f'Total Questions: {unique_questions}',  
    'answer_type': 'All',  
    'truth_answer': '',
    'predicted_answer': '',
    'accuracy': average_accuracy 
}
evaluation_results.append(average_result)

# Define column order with row_num first
column_order = [
    'row_num',
    'question_id',
    'question',
    'answer_type',
    'truth_answer',
    'predicted_answer',
    'accuracy'
]

# Create safe model name for file
safe_model_name = MODEL_NAME.replace('-', '_').replace('.', '_')

# Save to CSV
results_dir = os.path.join(os.getcwd(), "results")
os.makedirs(results_dir, exist_ok=True)
output_path = os.path.join(results_dir, f'simpsons_single_agent_{safe_model_name}.csv')

# First, check if the file exists and explicitly remove it
if os.path.exists(output_path):
    try:
        os.remove(output_path)
        print(f"Existing file removed: {output_path}")
    except Exception as e:
        print(f"Error removing existing file: {e}")

# Convert to DataFrame and save with error handling
try:
    results_df = pd.DataFrame(evaluation_results)
    results_df = results_df[column_order]
    
    # Save with explicit file opening to ensure it closes properly
    results_df.to_csv(output_path, index=False)
    
    # Verify the file was created
    if os.path.exists(output_path):
        print(f"Results successfully saved to: {output_path}")
    else:
        print(f"Warning: File was not created at {output_path}")
except Exception as e:
    print(f"Error saving results to CSV: {e}")

Existing file removed: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/simpsons_single_agent_claude_3_5_haiku_20241022.csv
Results successfully saved to: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/simpsons_single_agent_claude_3_5_haiku_20241022.csv
